<a href="https://colab.research.google.com/github/MohdAhmed627/StarSense---Multilingual-Review-Rating-System/blob/main/STARSENSE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install gradio transformers torch

Product Review Sentiment Analyzer
-----------------------------------
A modern, responsive Gradio web app that predicts a 1-5 star rating
for any product / movie / restaurant / service review using the
Hugging Face model: nlptown/bert-base-multilingual-uncased-sentiment

- Works directly in Google Colab or VS Code with no modification needed.

In [2]:
import gradio as gr
from transformers import pipeline

In [3]:
# ------------------------------------------------------------------
# 1. LOAD MODEL
# ------------------------------------------------------------------
# This model returns labels like "1 star", "2 stars", ... "5 stars"
# along with a confidence score.
print("Loading sentiment analysis model... please wait.")
classifier = pipeline(
    "text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)
print("Model loaded successfully!")

Loading sentiment analysis model... please wait.


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded successfully!


In [4]:

# ------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------------
def stars_from_label(label: str) -> int:
    """Extract the integer star count (1-5) from a label like '4 stars'."""
    return int(label.strip()[0])


def render_stars(num_stars: int) -> str:
    """Return a string like '★★★★☆' for the given number of stars (out of 5)."""
    num_stars = max(1, min(5, num_stars))
    return "★" * num_stars + "☆" * (5 - num_stars)


def predict_rating(review_text: str):
    """
    Takes a review string, runs it through the sentiment classifier,
    and returns nicely formatted markdown strings for:
      - the predicted star rating
      - the confidence score
    """
    # --- Input validation ---
    if review_text is None or review_text.strip() == "":
        error_msg = "⚠️ **Please enter a review before predicting.**"
        return error_msg, ""

    try:
        # --- Run prediction ---
        result = classifier(review_text)[0]   # e.g. {'label': '5 stars', 'score': 0.9987}
        label = result["label"]
        score = result["score"]

        num_stars = stars_from_label(label)
        star_display = render_stars(num_stars)
        confidence_pct = f"{score * 100:.2f}%"

        rating_output = (
            f"## {star_display}\n"
            f"### {num_stars} Star{'s' if num_stars != 1 else ''}"
        )
        confidence_output = f"## 📊 {confidence_pct}"

        return rating_output, confidence_output

    except Exception as e:
        error_msg = f"❌ **Something went wrong while predicting:** {str(e)}"
        return error_msg, ""


def clear_fields():
    """Reset the textbox and both output panels."""
    return "", "", ""


In [5]:
# ------------------------------------------------------------------
# 3. CUSTOM STYLING (Soft Blue / Modern Theme)
# ------------------------------------------------------------------
custom_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="sky",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Poppins"), "ui-sans-serif", "system-ui", "sans-serif"],
).set(
    button_primary_background_fill="*primary_500",
    button_primary_background_fill_hover="*primary_600",
    button_primary_text_color="white",
    block_radius="16px",
    block_shadow="0 4px 14px rgba(0, 0, 0, 0.08)",
    body_background_fill="*neutral_50",
)

custom_css = """
#title-text {
    text-align: center;
    font-weight: 800;
    font-size: 3rem;
    color: #1e40af;
    margin-bottom: 0.2rem;
}
#subtitle-text {
    text-align: center;
    color: #475569;
    font-size: 1.05rem;
    margin-bottom: 1.5rem;
}
#main-card {
    border-radius: 18px;
    padding: 20px;
}
#output-card {
    border-radius: 16px;
    padding: 18px;
    background: linear-gradient(135deg, #eff6ff, #f0f9ff);
    border: 1px solid #bfdbfe;
    text-align: center;
}
.gr-button {
    border-radius: 12px !important;
    font-weight: 600 !important;
}
footer {visibility: hidden}
"""


In [6]:

# ------------------------------------------------------------------
# 4. BUILD THE GRADIO INTERFACE (Blocks API)
# ------------------------------------------------------------------
with gr.Blocks(theme=custom_theme, css=custom_css, title="Review Sentiment Analyzer") as demo:

    # ---------- Header ----------
    gr.Markdown("StarSense", elem_id="title-text")
    gr.Markdown(
        "Multilingual Review Rating System <br>"
        " Enter any product, movie, restaurant, or service review to predict "
        "its star rating using a Hugging Face BERT model.",
        elem_id="subtitle-text",
    )

    # ---------- Main Layout ----------
    with gr.Column(elem_id="main-card"):
        with gr.Row():
            # ----- Input Column -----
            with gr.Column(scale=1):
                review_input = gr.Textbox(
                    label="✍️ Your Review",
                    placeholder="Type your review here...",
                    lines=8,
                    max_lines=12,
                )

                with gr.Row():
                    predict_btn = gr.Button("🔍 Predict Rating", variant="primary", scale=2)
                    clear_btn = gr.Button("🧹 Clear", variant="secondary", scale=1)

                gr.Examples(
                    examples=[
                        ["This phone is amazing. The battery lasts all day and the camera is excellent."],
                        ["The product arrived damaged and stopped working after two days."],
                        ["Average quality. It works fine but nothing special."],
                    ],
                    inputs=review_input,
                    label="💡 Try an example",
                )

            # ----- Output Column -----
            with gr.Column(scale=1, elem_id="output-card"):
                gr.Markdown("### ⭐ Predicted Rating")
                rating_output = gr.Markdown("_Your rating will appear here._")

                gr.Markdown("### 📊 Confidence Score")
                confidence_output = gr.Markdown("_Your confidence score will appear here._")

    # ---------- Footer ----------
    gr.Markdown(
        "<div style='text-align:center; color:#94a3b8; margin-top:1rem; font-size:0.85rem;'>"
        "Powered by Hugging Face Transformers • Model: nlptown/bert-base-multilingual-uncased-sentiment"
        "</div>"
    )

    # ---------- Event Handlers ----------
    predict_btn.click(
        fn=predict_rating,
        inputs=review_input,
        outputs=[rating_output, confidence_output],
    )

    clear_btn.click(
        fn=clear_fields,
        inputs=[],
        outputs=[review_input, rating_output, confidence_output],
    )


/tmp/ipykernel_525/1280436043.py:4: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css, title="Review Sentiment Analyzer") as demo:


In [7]:

# ------------------------------------------------------------------
# 5. LAUNCH APP
# ------------------------------------------------------------------
if __name__ == "__main__":
    demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0d089a161d90433a2b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
